# 🤖 PHẦN 4: TRÍCH CHỌN ĐẶC TRƯNG & MÔ HÌNH GỢI Ý (RECOMMENDATION ENGINE)
## File: `04_Feature_Engineering_And_Recommendation.ipynb`
---
**Mục tiêu của phần này:**
1. Đọc dữ liệu sạch từ `data/interim/cleaned_movies_2000_2026.csv`.
2. **Feature Engineering**:
   - Tính điểm xếp hạng có trọng số IMDb Bayesian Weighted Rating (WR).
   - Tạo chuỗi đặc trưng tổng hợp (`combined_features`) từ Tên phim + Thể loại (nhân đôi trọng số).
3. **Xây dựng mô hình Gợi Ý Lọc Dựa Trên Nội Dung (Content-Based Filtering)**:
   - Xây dựng ma trận TF-IDF Vectorizer (`max_features=5000`).
   - Tính độ tương đồng Cô-sin (Cosine Similarity Matrix).
4. Thử nghiệm hàm gợi ý phim tương tự khi người dùng nhập tên phim.
5. Xuất dữ liệu hoàn chỉnh ra `data/processed/movies_20000_2000_2026.csv` và `.json` để tích hợp vào Web App.

### 1. Khai báo thư viện & Nạp dữ liệu

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('data/interim/cleaned_movies_2000_2026.csv')
print(f'✅ Đã nạp thành công {len(df):,} bộ phim!')

### 2. Feature Engineering: Tính điểm Bayesian Weighted Rating (WR)
Công thức chuẩn hóa của IMDb:
$$WR = \left( \frac{v}{v + m} \right) \times R + \left( \frac{m}{v + m} \right) \times C$$
- $v$: Số lượt vote của phim
- $m$: Ngưỡng vote phân vị 70%
- $R$: Điểm rating trung bình của phim
- $C$: Điểm trung bình toàn bộ 20.000 phim

In [ ]:
C = df['rating'].mean()
m = df['votes'].quantile(0.70)

def bayesian_weighted_rating(row):
    v = row['votes']
    R = row['rating']
    return (v / (v + m)) * R + (m / (v + m)) * C

df['weighted_rating'] = df.apply(bayesian_weighted_rating, axis=1).round(2)

print(f'Điểm trung bình (C): {C:.2f}')
print(f'Ngưỡng vote phân vị 70% (m): {m:,.0f} votes')
display(df.sort_values(by='weighted_rating', ascending=False)[['title', 'year', 'rating', 'votes', 'weighted_rating']].head(5))

### 3. Tạo chuỗi đặc trưng tổng hợp (`combined_features`)
Ghép Tên phim và Thể loại để làm đầu vào cho thuật toán TF-IDF.

In [ ]:
# Ghép chuỗi đặc trưng: Tên phim + Thể loại (lặp 2 lần để tăng trọng số thể loại)
df['combined_features'] = df['title'] + ' ' + df['genres_clean'] + ' ' + df['genres_clean']
print('Ví dụ chuỗi đặc trưng của phim đầu tiên:')
print(df['combined_features'].iloc[0])

### 4. Xây dựng ma trận TF-IDF & Hàm Gợi Ý Cosine Similarity

In [ ]:
print('⏳ Đang tính toán ma trận TF-IDF cho 20.000 phim...')
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['combined_features'])
print(f'✅ Kích thước ma trận đặc trưng TF-IDF: {tfidf_matrix.shape}')

# Tạo tra cứu vị trí index theo tên phim (không phân biệt chữ hoa/thường)
indices = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()

def recommend_movies(movie_title, top_n=5):
    query = movie_title.lower().strip()
    if query not in indices:
        return f"Không tìm thấy phim '{movie_title}' trong kho 20.000 phim."
    
    idx = indices[query]
    # Tính độ tương đồng giữa phim được chọn và toàn bộ kho phim
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    # Lấy top các phim có độ tương đồng cao nhất (bỏ qua chính nó)
    top_idx = sim_scores.argsort()[::-1][1:top_n+1]
    
    result = df.iloc[top_idx][['title', 'year', 'genres', 'rating', 'weighted_rating']].copy()
    result['similarity_score'] = sim_scores[top_idx].round(3)
    return result

print('Đã khởi tạo xong hàm gợi ý phim!')

### 5. Thử nghiệm hàm gợi ý trên các bộ phim kinh điển

In [ ]:
print("🎯 GỢI Ý CÁC PHIM TƯƠNG TỰ VỚI 'Inception (2010)':")
display(recommend_movies('Inception', top_n=5))

print("\n🎯 GỢI Ý CÁC PHIM TƯƠNG TỰ VỚI 'The Dark Knight (2008)':")
display(recommend_movies('The Dark Knight', top_n=5))

print("\n🎯 GỢI Ý CÁC PHIM TƯƠNG TỰ VỚI 'Interstellar (2014)':")
display(recommend_movies('Interstellar', top_n=5))

### 6. Xuất tập dữ liệu sạch hoàn chỉnh ra `data/processed/`
Lưu dataset chuẩn cuối cùng để Backend FastAPI và Web App sử dụng trực tiếp.

In [ ]:
output_dir = 'data/processed'
os.makedirs(output_dir, exist_ok=True)

final_csv_path = os.path.join(output_dir, 'movies_20000_2000_2026.csv')
final_json_path = os.path.join(output_dir, 'movies_20000_2000_2026.json')

df.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
df.to_json(final_json_path, orient='records', force_ascii=False, indent=2)

print(f'✅ Đã xuất dataset CSV hoàn chỉnh: {final_csv_path} ({os.path.getsize(final_csv_path)/1024/1024:.2f} MB)')
print(f'✅ Đã xuất dataset JSON hoàn chỉnh: {final_json_path} ({os.path.getsize(final_json_path)/1024/1024:.2f} MB)')
print('\n🎉 CHÚC MỪNG BẠN! TOÀN BỘ PIPELINE TỪ CRAWL -> LÀM SẠCH -> EDA -> MODEL GỢI Ý ĐÃ HOÀN TẤT!')